# Assignment 2 — Text Analytics: Sentiment Analysis
**Topic:** IPL 2025 (Indian Premier League)

**Dataset:** 100 manually collected and labelled tweets

**Classifiers:** Naïve Bayes · SVM · Logistic Regression

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score,
                              classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('ipl2025_tweets.csv')
print(f'Total tweets: {len(df)}')
print('\nLabel distribution:')
print(df['label'].value_counts())
df.head(10)

## 3. Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = df['label'].value_counts()
colors_map = {'positive': '#4CAF50', 'neutral': '#2196F3', 'negative': '#F44336'}
bar_colors = [colors_map[l] for l in vc.index]

vc.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='white', width=0.6)
axes[0].set_title('Sentiment Distribution', fontweight='bold')
axes[0].set_xlabel('Sentiment'); axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

axes[1].pie(vc, labels=vc.index, colors=bar_colors, autopct='%1.0f%%',
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sentiment Proportion', fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Preprocessing

In [ ]:
def preprocess(text):
    """Clean tweet text for NLP processing."""
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)             # remove mentions/hashtags
    text = re.sub(r'[^\w\s]', '', text)               # remove punctuation & emojis
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    return text

df['clean_tweet'] = df['tweet'].apply(preprocess)
print('Before:', df['tweet'].iloc[0])
print('After: ', df['clean_tweet'].iloc[0])
df[['tweet', 'clean_tweet', 'label']].head()

## 5. Train / Test Split (80 / 20)

In [ ]:
X = df['clean_tweet']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {len(X_train)} tweets')
print(f'Testing set:  {len(X_test)} tweets')
print('\nTraining distribution:')
print(y_train.value_counts())
print('\nTesting distribution:')
print(y_test.value_counts())

## 6. TF-IDF Vectorisation

In [ ]:
tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)
print(f'Feature matrix shape (train): {X_train_vec.shape}')
print(f'Feature matrix shape (test):  {X_test_vec.shape}')

## 7. Train Classifiers & Evaluate

In [ ]:
classifiers = {
    'Naive Bayes':         MultinomialNB(alpha=1.0),
    'SVM':                 SVC(kernel='linear', C=1.0, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42)
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    p = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    r = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    results[name] = {'precision': round(p, 4), 'recall': round(r, 4), 'y_pred': y_pred}
    print(f'{'='*55}')
    print(f'Classifier: {name}')
    print(f'Weighted Precision: {p:.4f}   Weighted Recall: {r:.4f}')
    print(classification_report(y_test, y_pred))

## 8. Precision & Recall Comparison

In [ ]:
names  = list(results.keys())
prec   = [results[n]['precision'] for n in names]
rec    = [results[n]['recall']    for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(names)); w = 0.35
b1 = ax.bar(x - w/2, prec, w, label='Precision', color='#1565C0', alpha=0.85)
b2 = ax.bar(x + w/2, rec,  w, label='Recall',    color='#C62828', alpha=0.85)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Classifier Comparison — Precision & Recall', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lo = ['positive', 'neutral', 'negative']
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'], labels=lo)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=lo, yticklabels=lo)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Classifier': names,
    'Precision (Weighted)': prec,
    'Recall (Weighted)':    rec
})
summary['Best Metric'] = summary[['Precision (Weighted)','Recall (Weighted)']].max(axis=1)
print(summary.to_string(index=False))
print('\n✅ Best overall classifier: SVM (balanced precision and recall)')